# 21 - T-cell Gene-Set Validation Panel: CIDER vs 5 Independent RNA Signatures

**Purpose**: Supplemental validation figure with two panels. **Panel A**
correlates the DNA-based CIDER T-cell Score against five independent,
published RNA gene-set scores, grouped into three biological categories --
T-cell infiltration/presence, T-cell effector/cytotoxic activity, and
PD-L1 expression -- to show the CIDER T-cell Score tracks T-cell biology
consistently across many independently-derived signatures, not just the
one primary T-cell panel used elsewhere in the paper. **Panel B** checks
that this isn't just a correlation artifact by testing whether the DNA-based
CIDER score and the same five RNA gene sets actually discriminate ICI
responders from non-responders, the same way Figure 3 does for the primary
CIDER T-cell Score alone.

**Gene sets** (all computed in `0_prepare/2_rna.ipynb`; see that notebook
for exact gene lists, citations, and verification sources):

| Category | Gene set | Column | Already used elsewhere? |
|---|---|---|---|
| Infiltration | Primary T-cell signature (CD3D/CD3E/CD3G; Braun et al. 2021) | `tcell_tpm` | Yes -- `4_cider_validation.ipynb`, `19_dna_vs_rna_response.ipynb` |
| Infiltration | T-cell signature (HCL; Han et al. 2020) | `rna_tcell_hcl_score` | Yes -- `19_dna_vs_rna_response.ipynb` |
| Effector | CD8 Effector, tGE8 (Mariathasan et al. 2018 / Rosenberg et al. 2016) | `rna_teff_score` | Yes -- `19_dna_vs_rna_response.ipynb` |
| Effector | T-cell-inflamed GEP, Ayers 2017* | `rna_ayers_geps_score` | No -- new to this figure |
| PDL1 | PD-L1 (CD274) | `rna_pdl1_score` | Yes -- `19_dna_vs_rna_response.ipynb` |

Two other well-known published panels (Danaher et al. 2017, Bindea et al.
2013) remain deliberately **not** included -- their exact gene membership
could not be verified from a primary source in this project (see
`fig_references/tcell_signatures_literature.md` for the full verification
history). The T-cell-inflamed GEP above (marked `*`) is the one exception
in this figure: it IS included, but its 18-gene list is itself a
secondary-source reconstruction rather than one confirmed directly against
Ayers et al. 2017's own supplementary table -- every other gene set in the
table above has a list traceable to a primary source.

**Population:**
- **Panel A**: all Hartwig patients with RNA-seq (not restricted to
  ICI-treated), matching `4_cider_validation.ipynb`'s main RNA validation
  population -- broader than Panel B's ICI-only scope, since Panel A is
  about general RNA-DNA concordance across gene sets, not response/treatment
  specificity. Kept broad deliberately (not narrowed to match Panel B) for
  maximum correlation power.
- **Panel B**: ICI-treated patients with RNA-seq and known RECIST response
  (same `cpi_ready.rds` population convention as
  `19_dna_vs_rna_response.ipynb`'s Section 0), restricted further to
  patients with all six metrics (DNA-based CIDER score + all five RNA gene
  sets) present, so every box describes the exact same patient set.

**Method:**
- **Panel A**: Pearson correlation (`cor.test()`) between `log2(CIDER
  T-cell Score + 1)` and `log2(gene set score + 1)` for each of the 5 gene
  sets independently. Bars are coloured by category and ordered by category
  then by r (descending) within category; each bar is labelled with its
  Pearson r only (no separate error bar or significance annotation) -- the
  maximum 95% CI margin of error (Fisher z-transform) across all five gene
  sets is reported once, in the plot caption.
- **Panel B**: Styled like Figure 3 (`main_response_figure`, see
  `response_plot()` in `helpers/themes.r`) and like
  `19_dna_vs_rna_response.ipynb`'s own response boxplot -- one panel, one
  shared log-scale y-axis, with the DNA-based CIDER T-cell Score and the five
  RNA gene sets as x-axis categories and Responder/Non-Responder as the
  dodged fill group. Metrics are ordered by two-sided Wilcoxon significance
  (not FDR-corrected -- 6 tests, reported as plain p-values, same
  convention as `19_dna_vs_rna_response.ipynb`'s boxplot), left = least
  significant (highest p) to right = most significant (lowest p). Each
  metric is also labelled with its AUROC for Responder vs Non-Responder
  (`pROC::roc()`/`pROC::auc()`); for the five RNA gene sets this label also
  reports a DeLong's test (`pROC::roc.test(method = "delong")`) comparing
  that gene set's ROC curve to the DNA-based CIDER T-cell Score's ROC curve
  on the same patient set -- same DeLong convention used for the
  CIDER-vs-TMB comparisons in `9_roc.ipynb`. No ROC curves are plotted here
  (see `9_roc.ipynb`/`supplement_roc.png` for curve plots); only the summary
  AUROC/DeLong labels are shown.

**Input:** `ready.rds` (Panel A), `cpi_ready.rds` (Panel B)
**Output:** `figs/supplement_tcell_geneset_panel.rds`, `figs/supplement_tcell_geneset_panel.png` (Panels A/B, with AUROC/DeLong labels on Panel B)


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

library(ggplot2)
library(ggpubr)
library(patchwork)
library(pROC)

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)


## 1 — Data preparation


In [ ]:
# All Hartwig patients with RNA-seq -- same population convention as
# 4_cider_validation.ipynb's main RNA panel. RNA columns are sample-level
# and identical across every gp row for a given sample (same note as
# 19_dna_vs_rna_response.ipynb), so restrict to the T-cell row to avoid
# counting each sample up to 4 times (once per CDR3 locus-group).
# ready.rds also duplicates every row once more with cohort_broad forced to
# "ICI-Treated" (4_base.ipynb's bind_rows call, for Figure 3/4-style pooled
# panels) -- that duplication applies to every patient, not just ICI-treated
# ones, so it silently doubled this panel's sample size and biased every
# correlation. distinct(sampleId) collapses it back to one row per patient
# (RNA/CIDER values are identical between the two copies; only cohort_broad
# differs, which this notebook doesn't use for Panel A anyway).
ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE)

# One row per gene set: display label, category, and the column in `ready`
# holding that gene set's score. Order within this table is the DEFAULT
# category grouping (infiltration -> effector -> pdl1); final bar order
# is re-derived from the fitted r values below, not from this table's order.
# Labels here are the single source of truth for each gene set's display
# name -- Panel B's box_data_raw recode (Section 5) and auc_results'
# RNA_METRIC_COLS use these exact same strings so the two panels always show
# identical gene-set names (and so auc_results' later join onto box_data by
# metric name actually matches).
GENE_SET_INFO <- tibble::tribble(
  ~category,       ~label,                                        ~column,
  "Infiltration",  "T-cell (CD3D/CD3E/CD3G)\nBraun et al. 2021",   "tcell_tpm",
  "Infiltration",  "T-cell (HCL)\nHan et al. 2020",                "rna_tcell_hcl_score",
  "Effector",      "T-cell Effector (tGE8)\nMariathasan et al. 2018", "rna_teff_score",
  # Secondary-source reconstruction, not primary-source-verified -- see
  # 2_rna.ipynb's panel definition and
  # fig_references/tcell_signatures_literature.md. Label carries a "*" so
  # this lower verification confidence is visible on the plot itself.
  "Effector",      "T-cell Inflamed GEP*\nAyers 2017",             "rna_ayers_geps_score",
  "PDL1",          "PD-L1\n(CD274)",                              "rna_pdl1_score"
) |>
  dplyr::mutate(category = factor(category, levels = c("Infiltration", "Effector", "PDL1")))


## 2 — Compute correlations


In [ ]:
# Pearson correlation on the log2p1 scale, fit independently for each gene
# set against the same CIDER T-cell Score exposure (cider_score_depth_adj).
# (Pearson r is invariant to this choice of log base -- log2 vs log10 is
# just a linear rescaling of the same transformed values -- used here for
# consistency with the log2(x+1) convention used for CIDER/TMB elsewhere
# in the pipeline, e.g. 9_roc.ipynb.)
MIN_N_GENESET <- 20

fit_geneset_cor <- function(column, label, category) {
  d <- ready |>
    dplyr::transmute(
      x = log2(cider_score_depth_adj + 1),
      y = log2(.data[[column]] + 1)
    ) |>
    tidyr::drop_na(x, y)

  if (nrow(d) < MIN_N_GENESET) {
    message("Skipping ", label, " (n=", nrow(d), ")")
    return(NULL)
  }

  ct <- stats::cor.test(d$x, d$y, method = "pearson")
  tibble::tibble(
    category  = category,
    label     = label,
    column    = column,
    n         = nrow(d),
    r         = unname(ct$estimate),
    conf.low  = ct$conf.int[1],
    conf.high = ct$conf.int[2],
    p.value   = ct$p.value
  )
}

geneset_cor_results <- purrr::pmap_dfr(
  GENE_SET_INFO,
  function(category, label, column) fit_geneset_cor(column, label, category)
) |>
  dplyr::arrange(category, dplyr::desc(r))

message("\nGene-set correlations with CIDER T-cell Score (log2p1 scale, Pearson):")
print(geneset_cor_results |> dplyr::select(category, label, n, r, conf.low, conf.high, p.value))


## 3 — Panel A: Bar plot


In [ ]:
# Bars ordered by category, then by r (descending) within category -- the
# arrange() call above already put geneset_cor_results in this order, so
# just take that row order as the factor level order directly.
label_order <- geneset_cor_results$label

plot_df <- geneset_cor_results |>
  dplyr::mutate(label = factor(label, levels = rev(label_order)))

# Single grey fill, varying by alpha rather than hue -- Infiltration is the
# primary/best-established category (opaque), Effector and PDL1 step
# down in alpha so the eye reads "more established -> less established"
# rather than three unrelated colours.
category_alphas <- c(
  "Infiltration" = 1.00,
  "Effector"     = 0.70,
  "PDL1"         = 0.40
)

n_patients <- max(geneset_cor_results$n, na.rm = TRUE)

# Margin of error (half-width) of each gene set's 95% CI (Fisher
# z-transform, cor.test()) -- reported once as a single max-across-gene-sets
# figure in the caption below, rather than printing every CI on the bars
# themselves (which crowded the plot once five long gene-set labels were
# added). The bound is conservative: EVERY gene set's r is within this
# distance of its own CI edges, not just the worst one.
max_moe <- max(pmax(plot_df$r - plot_df$conf.low, plot_df$conf.high - plot_df$r))

p_tcell_geneset_panel <- plot_df |>
  ggplot(aes(x = label, y = r, alpha = category)) +
  geom_col(fill = "grey30", width = 0.7, colour = "grey30", linewidth = 0.3) +
  # Label each bar with its Pearson r only -- hjust flips outward depending
  # on the sign of r so the label always sits just past the end of the bar
  # rather than over it. 95% CIs are no longer shown per-bar (see caption
  # for the max margin of error across all five gene sets instead).
  # alpha = 1 (a fixed parameter, not an aes()) overrides the plot-level
  # aes(alpha = category) this layer would otherwise inherit from ggplot()
  # above -- every label reads as solid dark COL_INK regardless of its own
  # bar's category alpha, rather than fading along with Effector/PDL1.
  geom_text(
    aes(label = sprintf("%.2f", r),
        hjust = ifelse(r >= 0, -0.05, 1.05)),
    alpha = 1, size = 3.6, colour = COL_INK, show.legend = FALSE
  ) +
  geom_hline(yintercept = 0, colour = "grey50", linewidth = 0.3) +
  coord_flip() +
  scale_alpha_manual(values = category_alphas, name = "Category") +
  scale_y_continuous(expand = expansion(mult = c(0.12, 0.12))) +  # room for the on-bar labels
  go_theme +
  theme(
    axis.text.y  = element_text(size = 8),
    plot.title   = element_text(size = 13, face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5),
    # Left-justified (hjust = 0, ggplot's default is right-aligned, hjust =
    # 1), same caption convention used for Panel B and
    # 12_xgboost_model.ipynb's combined_xgb.
    plot.caption  = element_text(hjust = 0, size = 7.5, colour = COL_NEGATIVE,
                                 margin = margin(t = 6))
  ) +
  labs(
    x        = NULL,
    y        = "Pearson r vs. DNA-based CIDER T-cell Score (log2p1 scale)",
    title    = "CIDER T-cell Score vs 5 Independent RNA Gene-Set Signatures",
    subtitle = paste0("All Hartwig patients with RNA-seq, n = ", format(n_patients, big.mark = ","),
                       " (bars grouped by category, ordered by r within category)"),
    caption  = sprintf("Max 95%% CI margin of error across all five gene sets: \u00b1%.2f", max_moe)
  ) +
  # tags_main (helpers/themes.r) bundles the plot.tag theme element with a
  # stat_cor() Pearson-correlation annotation meant for continuous-x scatter
  # panels. Not applicable here (this bar chart's y-axis, label, is
  # categorical) -- and since that stat_cor() layer also inherits this
  # plot's aes(alpha = category) without setting show.legend = FALSE, it was
  # adding a stray text-glyph ("a") key to the Category legend on top of
  # being a meaningless annotation. Only the tag theme element is needed
  # here, so it's inlined directly instead of pulling in tags_main's
  # stat_cor layer too.
  theme(plot.tag = element_text(size = 11, face = "bold", margin = margin(b = 8, r = 8)))

set_plot_dims(9, 6.5); p_tcell_geneset_panel


## 4 — Panel B: Data preparation (ICI-treated, response)


In [ ]:
# ICI-treated patients with RNA-seq and known RECIST response -- same
# population convention as 19_dna_vs_rna_response.ipynb's Section 0:
# cpi_ready.rds carries a duplicated "ICI-Treated"-pooled copy of every
# ICI-treated row (4_base.ipynb) for Figure 3-style pooled-vs-per-cancer-type
# panels, which this figure doesn't use -- drop it first so patients aren't
# double-counted, then restrict to the T-cell row (RNA columns are
# sample-level and identical across every gp row for a given sample).
ici_ready <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(cohort_broad != "ICI-Treated", gp == "T-cell")


## 5 — Panel B: Boxplot


In [ ]:
# One box per metric: DNA-based CIDER T-cell Score plus all 5 RNA gene
# sets. Restricted to patients with every metric + responder present, so
# all six boxes (and the AUROC/DeLong comparisons below) describe the exact
# same patient set.
response_wide <- ici_ready |>
  dplyr::filter(!is.na(cider_score_depth_adj), !is.na(tcell_tpm),
                !is.na(rna_tcell_hcl_score), !is.na(rna_teff_score),
                !is.na(rna_ayers_geps_score),
                !is.na(rna_pdl1_score), !is.na(responder)) |>
  dplyr::select(sampleId, responder, cider_score_depth_adj, tcell_tpm,
                rna_tcell_hcl_score, rna_teff_score, rna_ayers_geps_score,
                rna_pdl1_score)

box_data_raw <- response_wide |>
  tidyr::pivot_longer(
    cols      = c(cider_score_depth_adj, tcell_tpm, rna_tcell_hcl_score,
                  rna_teff_score, rna_ayers_geps_score,
                  rna_pdl1_score),
    names_to  = "metric",
    values_to = "value"
  ) |>
  dplyr::mutate(
    # Same exact label strings as GENE_SET_INFO (Section 1) / RNA_METRIC_COLS
    # below, so gene-set names read identically in Panel A and Panel B.
    metric = dplyr::recode(metric,
      cider_score_depth_adj = "CIDER T-cell Score",
      tcell_tpm              = "T-cell (CD3D/CD3E/CD3G)\nBraun et al. 2021",
      rna_tcell_hcl_score    = "T-cell (HCL)\nHan et al. 2020",
      rna_teff_score         = "T-cell Effector (tGE8)\nMariathasan et al. 2018",
      rna_ayers_geps_score   = "T-cell Inflamed GEP*\nAyers 2017",
      rna_pdl1_score         = "PD-L1\n(CD274)"
    )
  )

n_panel_b <- box_data_raw |> dplyr::distinct(sampleId) |> nrow()

# Order metrics by statistical significance (two-sided Wilcoxon, Responder
# vs Non-Responder) rather than by biological category -- left = least
# significant (highest p), right = most significant (lowest p). This is the
# same per-metric Wilcoxon test `ggpubr::stat_compare_means()` draws on the
# plot below, just computed up front here to fix the metric order.
metric_wilcox_p <- box_data_raw |>
  dplyr::group_by(metric) |>
  dplyr::summarise(p = wilcox.test(value ~ responder)$p.value, .groups = "drop") |>
  dplyr::arrange(dplyr::desc(p))

box_data <- box_data_raw |>
  dplyr::mutate(metric = factor(metric, levels = metric_wilcox_p$metric))

# ── AUROC + DeLong's test vs the DNA-based CIDER T-cell Score ─────────────────
# Same response_wide population used for every box above (all six metrics +
# responder present), so every AUROC comes from the exact same N and is
# directly comparable. AUC/ROC-curve shape is invariant to any monotonic
# transform of the predictor (e.g. log2(x+1)) -- it only depends on how
# patients are RANKED by the predictor -- so raw scores are used directly;
# no transform is needed just to compute AUROC. DeLong's test
# (pROC::roc.test(), same method used for the CIDER-vs-TMB comparisons in
# 9_roc.ipynb) compares each RNA gene set's ROC curve against the DNA-based
# CIDER T-cell Score's ROC curve on this same patient set. No ROC curves are
# drawn here (see 9_roc.ipynb/supplement_roc.png for curve plots) -- only
# the summary AUROC/DeLong labels below. The DNA score itself has no DeLong
# comparison (it's the reference, not compared to itself).
response_bin <- as.integer(response_wide$responder == "Responder")
roc_cider    <- pROC::roc(response_bin, response_wide$cider_score_depth_adj, quiet = TRUE)

# Must match box_data_raw's recode() above character-for-character --
# auc_results is later joined onto box_data by this exact metric string.
RNA_METRIC_COLS <- c(
  tcell_tpm            = "T-cell (CD3D/CD3E/CD3G)\nBraun et al. 2021",
  rna_tcell_hcl_score  = "T-cell (HCL)\nHan et al. 2020",
  rna_teff_score       = "T-cell Effector (tGE8)\nMariathasan et al. 2018",
  rna_ayers_geps_score = "T-cell Inflamed GEP*\nAyers 2017",
  rna_pdl1_score       = "PD-L1\n(CD274)"
)

auc_results <- purrr::imap_dfr(RNA_METRIC_COLS, function(metric_label, col) {
  roc_metric <- pROC::roc(response_bin, response_wide[[col]], quiet = TRUE)
  delong_p   <- pROC::roc.test(roc_cider, roc_metric, method = "delong")$p.value
  tibble::tibble(
    metric   = metric_label,
    auc      = as.numeric(pROC::auc(roc_metric)),
    delong_p = delong_p
  )
}) |>
  dplyr::bind_rows(
    tibble::tibble(
      metric   = "CIDER T-cell Score",
      auc      = as.numeric(pROC::auc(roc_cider)),
      delong_p = NA_real_
    )
  ) |>
  dplyr::mutate(metric = factor(metric, levels = levels(box_data$metric)))

# fmt_pval() is scalar-only (its `if (is.na(p))` isn't vectorised) -- ifelse()
# would evaluate fmt_pval(delong_p) across the whole column at once
# (including the TRUE branch for CIDER's NA row) and error with "the
# condition has length > 1". map2_chr() calls it once per row instead.
auc_results <- auc_results |>
  dplyr::mutate(
    auroc_label = purrr::map2_chr(auc, delong_p, function(a, p) {
      if (is.na(p)) {
        sprintf("AUC = %.2f", a)
      } else {
        sprintf("AUC = %.2f\nDeLong %s", a, fmt_pval(p))
      }
    })
  )

# Per-metric max, used only to anchor the AUROC/DeLong label (the one
# addition versus the standard boxplot below) a bit above each box -- same
# "anchor to this group's own data, nudge with vjust" idiom response_plot()
# already uses for its median/count labels, just applied to a new label.
# Deliberately NOT a global/shared anchor across metrics and NOT a
# log-space fixed-offset calculation -- both of those were tried and made
# things worse. Per-metric, vjust-nudged, exactly like everything else on
# this plot.
metric_max <- box_data |>
  dplyr::group_by(metric) |>
  dplyr::summarise(y_max = max(value, na.rm = TRUE), .groups = "drop")

auc_results <- auc_results |>
  dplyr::left_join(metric_max, by = "metric")

# Jitter data: cap at 150 points per metric x response group to avoid
# overplotting -- same cap as helpers/themes.r's response_plot().
jitter_data <- box_data |>
  dplyr::group_by(metric, responder) |>
  dplyr::slice_sample(n = 150) |>
  dplyr::ungroup()

# Per-group medians (shown above each box) and counts (shown at the bottom)
# -- identical to response_plot()'s own med_labels/count_labels.
med_labels <- box_data |>
  dplyr::group_by(metric, responder) |>
  dplyr::summarise(med = median(value, na.rm = TRUE), .groups = "drop") |>
  dplyr::mutate(med_label = as.character(round(med, 1)))

count_labels <- box_data |>
  dplyr::group_by(metric, responder) |>
  dplyr::summarise(n = dplyr::n(), .groups = "drop") |>
  dplyr::mutate(n_label = paste0("n = ", n))

# Same design as every other response boxplot in this paper
# (helpers/themes.r's response_plot(), used by 1_response.ipynb/Figure 3,
# 17_response_by_read_confidence.ipynb, 19_dna_vs_rna_response.ipynb): one
# panel, one shared log-scale y-axis, metric on the x-axis dodged by
# responder, background jitter, boxplot, median label, count label, and a
# plain (default-positioned) Wilcoxon p-value per metric via
# stat_compare_means(). The ONLY addition versus that shared design is the
# AUROC/DeLong geom_text layer below -- no faceting, no custom axis breaks,
# no custom label-height system; six metrics on six different raw scales
# simply read differently (not a bug), the same way Figure 3's own
# response_plot() shows very differently-scaled/sized cohorts side by side
# on one shared axis without issue.
p_response_geneset_panel <- box_data |>
  ggplot(aes(x = metric, y = value, fill = responder)) +
  geom_jitter(
    data     = jitter_data,
    aes(colour = responder),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    alpha = 0.2, size = 0.5, show.legend = FALSE
  ) +
  scale_colour_manual(values = responder_colors, guide = "none") +
  geom_boxplot(alpha = 0.7, outlier.shape = NA) +
  geom_text(
    data        = med_labels,
    aes(x = metric, y = med, label = med_label, group = responder),
    inherit.aes = FALSE,
    position    = position_dodge(width = 0.75),
    vjust = -0.4, size = 2.2, fontface = "bold", colour = COL_INK
  ) +
  geom_text(
    data        = count_labels,
    aes(x = metric, y = 0, label = n_label, group = responder),
    inherit.aes = FALSE,
    position    = position_dodge(width = 0.75),
    vjust = 1.5, size = 2.1, colour = COL_NEGATIVE
  ) +
  scale_fill_manual(values = responder_colors) +
  # Fixed at label.y = log10(900) -- a literal transformed-space coordinate
  # (log10p1_trans's own `transform` is log10(x+1), so this is NOT the same
  # screen position as the raw value 900 would produce; confirmed/intended
  # by request).
  ggpubr::stat_compare_means(
    aes(group = responder), method = "wilcox.test",
    label = "p.format", digits = 2, label.y = log10(900), vjust = 0, colour = COL_INK
  ) +
  # The one addition versus response_plot(): AUROC/DeLong. Fixed at
  # y = log10(600) -- same literal transformed-space convention as the
  # Wilcoxon label.y above (confirmed/intended by request) -- rather than
  # the previous per-metric y_max anchor, so y is no longer mapped in aes().
  geom_text(
    data        = auc_results,
    aes(x = metric, label = auroc_label),
    inherit.aes = FALSE,
    y = log10(600), size = 3, lineheight = 0.85, colour = COL_INK
  ) +
  scale_x_discrete(guide = guide_axis(n.dodge = 2)) +
  scale_y_continuous(
    trans  = log10p1_trans,
    # Symmetric 4% padding both above and below the data range (was
    # asymmetric 4% / 18%), in log10p1-transformed space.
    expand = expansion(mult = c(0.04, 0.04))
  ) +
  go_theme +
  theme(
    plot.title    = element_text(size = 13, face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5)
  ) +
  labs(
    x        = NULL,
    y        = "Score (log scale)",
    fill     = "Response",
    title    = "CIDER and RNA Gene Sets -- Responder Discrimination",
    subtitle = paste0("ICI-treated patients with all 6 metrics present, n = ", format(n_panel_b, big.mark = ","))
  ) +
  # tags_main (helpers/themes.r) bundles the plot.tag theme element with a
  # stat_cor() Pearson-correlation annotation meant for continuous-x scatter
  # panels -- not applicable to this categorical-x (metric) boxplot, and
  # would just add a meaningless "R = ..." label on top of everything else.
  # Only the tag theme element is needed here, so it's inlined directly
  # instead of pulling in tags_main's stat_cor layer too.
  theme(plot.tag = element_text(size = 11, face = "bold", margin = margin(b = 8, r = 8)))

set_plot_dims(12, 5.3); p_response_geneset_panel


## 6 — Assemble and save


In [ ]:
# Panel A (bar plot, Section 3) stacked above Panel B (response boxplot,
# Section 5) -- patchwork's figure_annotation()/plot_annotation(tag_levels
# = "A") assigns the A/B letters automatically from this composition order.
supplement_tcell_geneset_panel <-
  p_tcell_geneset_panel / p_response_geneset_panel +
  plot_layout(heights = c(1.1, 1)) +
  figure_annotation("CIDER T-cell Score vs Independent RNA Gene-Set Signatures: Correlation and Response Discrimination")

set_plot_dims(11, 12); supplement_tcell_geneset_panel

saveRDS(supplement_tcell_geneset_panel, paste0(CIDER_PROJ_DIR, "figs/supplement_tcell_geneset_panel.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_tcell_geneset_panel.png"),
       supplement_tcell_geneset_panel, width = 11, height = 12, dpi = 300)
message("Saved: supplement_tcell_geneset_panel.rds and supplement_tcell_geneset_panel.png")
